# Backlog prediction with TabFM (Google)

Simple adaptation of the backlog prediction pipeline using [TabFM](https://github.com/google-research/tabfm), a zero-shot tabular foundation model (in-context learning), instead of the pipeline with custom encoders + RidgeCV.

References:
- https://research.google/blog/introducing-tabfm-a-zero-shot-foundation-model-for-tabular-data/
- https://github.com/google-research/tabfm

No SHAP and no manual encoders: TabFM natively handles mixed categorical and numerical columns.

In [12]:
import sys

sys.path.append('..')

from tabfm import TabFMRegressor
from tabfm import tabfm_v1_0_0_pytorch as tabfm_v1_0_0

from modules.data_ingestion import load_sheet, prepare_data

In [13]:
SHEET_NAME = 'Backlog'

# TabFM natively handles categoricals, so we pass the "raw" columns instead
# of the custom encoders used in the Ridge pipeline.
FEATURES: list[str] = [
    'Gênero',
    'Franquia',
    'Desenvolvedora',
    'Metacritic Score (AI)',
    'User Score (AI)',
]

STATUS_MULTIPLIERS: dict[str, float] = {
    '2. Próximos': 1.05,
    '4. Backlog': 1,
    '5. Rejogar': 0.95,
    '6. Dar Outra Chance': 0.9,
}

## 1. Load data and split finished games (train) vs backlog

In [14]:
df = load_sheet(SHEET_NAME, local=True)
finished, backlog = prepare_data(df)

X_train = finished[FEATURES]
y_train = finished['Nota']
X_backlog = backlog[FEATURES]

print(f'Finished games (train): {len(finished)}')
print(f'Backlog games (predict): {len(backlog)}')
X_train.head()

Finished games (train): 49
Backlog games (predict): 60


,Gênero,Franquia,Desenvolvedora,Metacritic Score (AI),User Score (AI)
0,"Gerenciamento, Cozy",Pokémon,Koei Tecmo,90.0,8.7
1,"RPG, Ação/Aventura",Zelda,Nintendo,97.0,8.9
2,JRPG,Final Fantasy,Square Enix,92.0,8.9
3,"Ação/Aventura, Coop",LEGO,TT Games,84.0,8.8
4,"Survival Horror, Coop",Resident Evil,Capcom,69.0,5.7


## 2. Load the pre-trained model and run zero-shot fit/predict

There's no gradient descent here: `fit` just stores the training rows as context, and `predict` uses in-context learning over that context.

In [15]:
# model_type='regression' is required: the default ('classification') loads
# a checkpoint with a 10-class head, incompatible with TabFMRegressor
# (raises a ValueError when trying to squeeze the output down to 1 value).
model = tabfm_v1_0_0.load(model_type='regression')

reg = TabFMRegressor(model=model)
reg.fit(X_train, y_train)

base_scores = reg.predict(X_backlog)
base_scores[:10]

array([8.887625 , 8.520497 , 7.9769077, 8.002035 , 8.812864 , 8.201729 ,
       8.321991 , 8.587807 , 8.140455 , 8.098588 ], dtype=float32)

## 3. Apply status multiplier and rank

In [16]:
result = backlog[
    [
        'Jogo',
        'Franquia',
        'Desenvolvedora',
        'Gênero',
        'Status',
        'Metacritic Score (AI)',
        'User Score (AI)',
    ]
].copy()
result['base_score'] = base_scores
result['multiplier'] = result['Status'].map(STATUS_MULTIPLIERS).fillna(1.0)
result['final_score'] = result['base_score'] * result['multiplier']
result = result.sort_values('final_score', ascending=False).reset_index(drop=True)

result

,Jogo,Franquia,Desenvolvedora,Gênero,Status,Metacritic Score (AI),User Score (AI),base_score,multiplier,final_score
0,Dark Souls 3 (replay),Souls,FromSoftware,"Coop, Soulslike",2. Próximos,89.0,8.7,8.887625,1.05,9.332006
1,Disco Elysium,Independente,ZA/UM,"CRPG, Aventura Narrativa",4. Backlog,97.0,8.3,9.081412,1.00,9.081412
2,The Legend of Zelda: Tears of the Kingdom,Zelda,Nintendo,"RPG, Ação/Aventura",4. Backlog,96.0,8.8,9.031619,1.00,9.031619
3,Resident Evil HD REMASTER,Resident Evil,Capcom,Survival Horror,2. Próximos,82.0,8.5,8.520497,1.05,8.946522
4,Dead as Disco,Independente,Brain Jar Games,Aventura Narrativa,4. Backlog,82.0,9.8,8.885112,1.00,8.885112
5,Kingdom Come: Deliverance II,Kingdom Come,Warhorse Studios,RPG,4. Backlog,88.0,8.7,8.812864,1.00,8.812864
6,"Warhammer 40,000: Space Marine 2",Warhammer 40k,Saber Interactive,"Shooter / Tiro, Hack and Slash",4. Backlog,82.0,7.9,8.812343,1.00,8.812343
7,The Riftbreaker,Independente,EXOR Studios,"Estratégia, Simulação / Arcade",4. Backlog,83.0,8.2,8.770313,1.00,8.770313
8,Prey,Independente,Arkane Studios,"Immersive Sim, Shooter / Tiro",4. Backlog,82.0,8.2,8.713820,1.00,8.713820
9,Gravity Circuilt,Independente,Domesticated Ant Games,Plataforma / Ação 2D,4. Backlog,89.0,7.9,8.703468,1.00,8.703468
